<h1 align="center">Play Memgraph with ORE Data</h1>

# Table of contents <a name="toc"></a>
1. [Introduction](#introduction)
2. [Prerequisites](#prerequisites)
3. [Requirements](#requirements)
4. [Source Dataset](#dataset)
5. [Connect to Memgraph with GQLAlchemy](#connect)
6. [Define a Graph Data Model](#graph-schema)
7. [Creating and returning nodes and relationships](#create-return)
8. [Importing data from CSV files](#import)
9. [Querying the database and retrieving results](#query)
10. [Use Case queries](#exercises)

## 1. Introduction <a name="introduction"></a>

Through this demo, we will create a graph model from a dataset, run Memgraph with Docker, connect to it from a Jupyter Notebook with the help of GQLAlchemy, and perform simple queries. 

## 2. Prerequisites <a name="prerequisites"></a>

For this demo, we need to install:

- [Jupyter](https://jupyter.org/install) - to run the Jupyter notebook
- [Docker](https://docs.docker.com/get-docker/) - to run Memgraph, since Memgraph is a native Linux application and cannot be installed on Windows and macOS
- [CMake](https://cmake.org/install/) - a prerequisite for GQLAlchemy
- [Memgraph Platform](https://memgraph.com/docs/memgraph/installation) - the complete streaming graph application platform; follow the instructions to install Memgraph Platform with Docker for your OS

> ### Memgraph Platform installation using Docker
>
> After we install Docker, we can run the Memgraph Platform container by running:
>
>```
>docker run -it -p 7687:7687 -p 7444:7444 -p 3000:3000 memgraph/memgraph-platform:2.4.0
>```
>
>**Memgraph Platform** contains:
>
>- **MemgraphDB** - the database that holds your data
>- **Memgraph Lab** - visual user interface for running queries and visualizing graph data (running at `localhost:3000`)
>- **mgconsole** - command-line interface for running queries
>- **MAGE** - graph algorithms and modules library

## 3. Requirements  <a name="requirements"></a>

For this demo, we need to install the following Python libraries:

- [GQLAlchemy](https://pypi.org/project/gqlalchemy/)
- [Pandas](https://pypi.org/project/pandas/)

All info about the versions can be found in the [requirements.txt] file.

## 4. Source Dataset <a name="dataset"></a>


In [1]:
import pandas as pd

site = pd.read_csv("../data/site.csv")
site.head()

,site_number,store_code,site_name,site_address,latitude,longitude
0,1001,WDAZ01,DTD PHOENIX AZ (WDAZ01),2300 S 51ST AVESTE 140,33.426933,-112.171425
1,1002,AZF 01,S MILTON (AZF 01),1230 S MILTON RD,35.188653,-111.661367
2,1003,AZP 01,EAST THOMAS (AZP 01),3234 E THOMAS RD,33.480606,-112.011124
3,1005,AZP 03,MAIN (AZP 03),1929 W MAIN ST,33.414660,-111.873261
4,1007,AZP 05,SOUTHERN (AZP 05),1709 E SOUTHERN AVE,33.391693,-111.911027


In [2]:
site.count()

site_number     1408
store_code      1408
site_name       1408
site_address    1408
latitude        1401
longitude       1401
dtype: int64

In [3]:
site['store_code_rank'] = site.groupby('site_number')['store_code'].rank(method='first')
df = site[site['store_code_rank']==1]
df = df.drop('store_code_rank', axis=1)
site.count()
# df.to_csv('../data/article_inventory.csv', index=False)

site_number        1408
store_code         1408
site_name          1408
site_address       1408
latitude           1401
longitude          1401
store_code_rank    1408
dtype: int64

In [4]:
article = pd.read_csv("../data/product.csv")
article.head()

,article_number,article_type_code,created_date,manufacturer_code,gross_weight,net_weight,speed_rating_code,tire_cross_section_number,tire_aspect_ratio,tire_rim_size_number,...,wheel_offset_number,wheel_diameter_number,wheel_style_code,wheel_bolt_pattern,wheel_accent_code,spoke_100mm_depth,spoke_120mm_depth,spoke_160mm_depth,spoke_90mm_depth,merchandise_category_code
0,10000,ZPRO,2014-09-03,NaN,0.0,0.00,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300056
1,10001,ZTIR,2014-09-15,FAL,54.0,53.50,Q,245,75.0,16.0,...,30.57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300001
2,10002,ZTIR,2014-09-15,FAL,55.0,54.30,S,265,70.0,17.0,...,31.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300001
3,10003,ZTIR,2014-08-22,NIT,32.0,31.97,W,245,55.0,18.0,...,28.54,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300000
4,10004,ZTIR,2014-09-15,FAL,61.0,60.50,Q,275,65.0,18.0,...,32.16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,300001


In [5]:
article.count()

article_number               50000
article_type_code            50000
created_date                 50000
manufacturer_code            49980
gross_weight                 50000
net_weight                   50000
speed_rating_code            28182
tire_cross_section_number    28832
tire_aspect_ratio            28626
tire_rim_size_number         28832
tread_depth                  28832
tire_load_capacity           28832
tire_load_index              28832
wheel_offset_number          28832
wheel_diameter_number        20915
wheel_style_code             20915
wheel_bolt_pattern           20915
wheel_accent_code            20915
spoke_100mm_depth            20915
spoke_120mm_depth            20915
spoke_160mm_depth            20915
spoke_90mm_depth             20915
merchandise_category_code    50000
dtype: int64

In [6]:
employee = pd.read_csv("../data/employee.csv")
employee.head()

,employee_identifier,site_number,full_name,employee_type_name,employment_status_code,position_effective_start_date,effective_termination_date,position_name,position_job_code,original_hire_date,create_timestamp
0,316846,2199,"Pickens, Jacob H",Full time,A,2024-08-01,NaN,Store Manager,1752,2000-10-09,2024-03-12 21:44:27
1,525040,2199,"Apodaca Cordova, Jesus A",Full time,A,2023-04-23,NaN,Store Sr Asst Manager,1772,2014-06-16,2024-03-12 21:44:31
2,602743,2199,"White, Jarrett T",Full time,A,2024-03-03,NaN,Store Asst Manager,1782,2020-02-12,2024-03-12 21:44:24
3,607724,2199,"Penrod, Joshua A",Full time,A,2024-06-23,NaN,Store Asst Manager,1782,2020-08-17,2024-03-12 21:44:29
4,644863,2199,"Keifer, Samule",Full time,A,2022-05-27,NaN,Store Asst Manager,1782,2022-05-27,2024-03-12 21:44:25


In [7]:
employee.count()

employee_identifier              45288
site_number                      45288
full_name                        45288
employee_type_name               45288
employment_status_code           45288
position_effective_start_date    45288
effective_termination_date       17341
position_name                    45288
position_job_code                45288
original_hire_date               45288
create_timestamp                 45288
dtype: int64

In [8]:
sales = pd.read_csv("../data/sales.csv")
sales.head()

,sales_order_identifier,create_employee_identifier,sales_order_created_date,article_number,sold_quantity,retail_price,discount_amount,net_price
0,800004001,NaN,2023-07-11,25769,1,354.0,0.0,354.0
1,800004001,NaN,2023-07-11,80007,1,1.0,-1.0,0.0
2,800004002,NaN,2023-08-03,128721,1,182.0,0.0,182.0
3,800004002,NaN,2023-08-03,91250,1,0.0,0.0,0.0
4,800005001,NaN,2023-07-11,80085,1,0.0,0.0,15.0


In [9]:
sales.count()

sales_order_identifier        50000
create_employee_identifier    39854
sales_order_created_date      50000
article_number                50000
sold_quantity                 50000
retail_price                  50000
discount_amount               50000
net_price                     50000
dtype: int64

In [10]:
sales['sales_order_created_date_rank'] = sales.groupby('sales_order_identifier')['sales_order_created_date'].rank(method='first')
df = sales[sales['sales_order_created_date_rank']==1]
df = df.drop('sales_order_created_date_rank', axis=1)
df.count()
df.to_csv('../data/sales_curated.csv', index=False)

In [11]:
inventory = pd.read_csv("../data/inventory.csv")
inventory.head()

,inventory_id,site_number,article_number,reserved_quantity,on_hand_quantity,available_quantity,in_transit_quantity,layaway_quantity,weborder_quantity,inventory_date
0,11082,1007,10075,0,0.0,0,NaN,NaN,NaN,2024-04-11
1,11164,1007,10157,4,0.0,-4,NaN,NaN,NaN,2024-07-17
2,11164,1007,10157,4,2.0,-2,NaN,NaN,NaN,2024-07-18
3,11164,1007,10157,4,4.0,0,NaN,NaN,NaN,2024-07-20
4,11164,1007,10157,0,0.0,0,NaN,NaN,NaN,2024-07-22


In [12]:
inventory.count()

inventory_id           50000
site_number            50000
article_number         50000
reserved_quantity      50000
on_hand_quantity       49924
available_quantity     50000
in_transit_quantity    20667
layaway_quantity       20667
weborder_quantity      20667
inventory_date         50000
dtype: int64

In [13]:
inventory['inventory_date_rank'] = inventory.groupby('inventory_id')['inventory_date'].rank(method='first')

In [14]:
df = inventory[inventory['inventory_date_rank']==1]
df = df.drop('inventory_date_rank', axis=1)
df.to_csv('../data/article_inventory.csv', index=False)

## 5. Connect to Memgraph with GQLAlchemy <a name="connect"></a>

We will use the **GQLAlchemy**'s object graph mapper (OGM) to connect to Memgraph and quickly execute **Cypher** queries. GQLAlchemy also serves as a Python driver/client for Memgraph. We can install it using:
```
pip install gqlalchemy==1.3.2
```
> You may need to install [CMake](https://cmake.org/download/) before installing GQLAlchemy.

In [15]:
from gqlalchemy import Memgraph

memgraph = Memgraph("127.0.0.1", 7687)
memgraph.drop_database()

In [16]:
results = memgraph.execute_and_fetch(
    """
    MATCH (n) RETURN count(n) AS number_of_nodes ;
    """
)
print(next(results))

{'number_of_nodes': 0}


## 6. Define a Graph Data Model <a name="graph-schema"></a>

Next, we have to define the above graph schema in the Python code. This is where **GQLAlchemy** steps in with its Object Graph Mapper.

In [17]:
from typing import Optional
from gqlalchemy import Node, Relationship, Field


class Site(Node):
    site_id: int = Field(index=True, exists=True, db=memgraph)
    store_code      : Optional[str]   = Field()
    site_name       : Optional[str]   = Field()
    site_address    : Optional[str]   = Field()
    lat             : Optional[float] = Field()
    lng             : Optional[float] = Field()


class Article(Node):
    article_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    article_type_code         : Optional[str] = Field()
    created_date              : Optional[str] = Field()
    manufacturer_code         : Optional[str] = Field()
    gross_weight              : Optional[str] = Field()
    net_weight                : Optional[str] = Field()
    speed_rating_code         : Optional[str] = Field()
    tire_cross_section_number : Optional[str] = Field()
    tire_aspect_ratio         : Optional[str] = Field()
    tire_rim_size_number      : Optional[str] = Field()
    tread_depth               : Optional[str] = Field()
    tire_load_capacity        : Optional[str] = Field()
    tire_load_index           : Optional[str] = Field()
    wheel_offset_number       : Optional[str] = Field()
    wheel_diameter_number     : Optional[str] = Field()
    wheel_style_code          : Optional[str] = Field()
    wheel_bolt_pattern        : Optional[str] = Field()
    wheel_accent_code         : Optional[str] = Field()
    spoke_100mm_depth         : Optional[str] = Field()
    spoke_120mm_depth         : Optional[str] = Field()
    spoke_160mm_depth         : Optional[str] = Field()
    spoke_90mm_depth          : Optional[str] = Field()
    merchandise_category_code : Optional[str] = Field()


class Employee(Node):
    employee_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    site_number                    : Optional[int] = Field()
    full_name                      : Optional[str] = Field()
    employee_type_name             : Optional[str] = Field()
    employment_status_code         : Optional[str] = Field()
    position_effective_start_date  : Optional[str] = Field()
    effective_termination_date     : Optional[str] = Field()
    position_name                  : Optional[str] = Field()
    position_job_code              : Optional[str] = Field()
    original_hire_date             : Optional[str] = Field()
    create_timestamp               : Optional[str] = Field()


class Inventory(Node):
    inventory_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    site_number            : Optional[int] = Field()
    article_number         : Optional[int] = Field()
    reserved_quantity      : Optional[int] = Field()
    on_hand_quantity       : Optional[int] = Field()
    available_quantity     : Optional[int] = Field()
    in_transit_quantity    : Optional[int] = Field()
    layaway_quantity       : Optional[int] = Field()
    weborder_quantity      : Optional[int] = Field()
    inventory_date         : Optional[str] = Field()


class Sales(Node):
    sales_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    employee_number               : Optional[int] = Field()
    article_number                : Optional[int] = Field()
    sales_order_created_date      : Optional[str] = Field()
    sold_quantity                 : Optional[int] = Field()
    retail_price                  : float = Field()
    discount_amount               : float = Field()
    net_price   				  : float = Field()


class OwnInventory(Relationship, type="OWN"):
    r_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    inventory_id       : Optional[int] = Field()
    site_id            : Optional[int] = Field()
    create_timestamp   : Optional[str] = Field()


class CreateSales(Relationship, type="CREATES"):
    r_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    employee_id        : Optional[int] = Field()
    sales_id           : Optional[int] = Field()
    create_timestamp   : Optional[str] = Field()


class BelongtoSite(Relationship, type="BELONGS"):
    r_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    employee_id        : Optional[int] = Field()
    site_id            : Optional[int] = Field()
    create_timestamp   : Optional[str] = Field()


class SellArticle(Relationship, type="SOLD"):
    r_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    sales_id           : Optional[int] = Field()
    article_id         : Optional[int] = Field()
    create_timestamp   : Optional[str] = Field()


class InventoryHoldArticle(Relationship, type="Hold"):
    r_id: int = Field(index=True, unique=True, exists=True, db=memgraph)
    inventory_id       : Optional[int] = Field()
    article_id         : Optional[int] = Field()
    create_timestamp   : Optional[str] = Field()

## 7. Creating and returning nodes and relationships <a name="create-return"></a>


In [18]:
# article_node = Article(id=10000, 
#                        article_description="YOKOHAMA TEAM REBATE", 
#                        article_type_code="ZPRO")
# article_node.save(memgraph)

# print(article_node)

In [19]:
# site_node = Site(id=100000,
#                  site_name="DTD PHOENIX AZ (WDAZ01)",
#                  site_address="2300 S 51ST AVESTE 140",
#                  site_state_code="AZ",
#                  lat=33.426933,
#                  lng=-112.171425)
# site_node.save(memgraph)
# print(site_node)

We can validate it from Memgraph Lab:
```
MATCH (n)
RETURN *;
```

Next, let's create a relationship of type `Sales` between the existing `Site` and `Article` nodes.

In [20]:
# sales_relationship = Sales(_start_node_id=site_node._id, 
#                            _end_node_id=article_node._id, 
#                            sold_quantity=100, 
#                            retail_price=988.90,
#                            sales_timestamp="2024-12-25 21:43:16")

# sales_relationship.save(memgraph)

# print(sales_relationship)

we can validate it from Memgraph Lab:
```
MATCH (n)-[r]->(m)
RETURN *;
```

## 8. Importing data from CSV files <a name="import"></a>

First, let's clean up the database to import only data from the CSV files.


In [21]:
memgraph.drop_database()

Now we nee to import `movies.csv` and `ratings.csv` files into Memgraph, by copying them from the [data folder](../data) into the Docker container where Memgraph is running. First, place yourself into the data folder. Next, find the `CONTAINER_ID` by running:

```
docker ps
```

Copy the files with the following commands (don't forget to replace `CONTAINER_ID`):

```
docker cp article_inventory.csv 3910733944a2:article_inventory.csv
docker cp employee.csv 3910733944a2:employee.csv
docker cp product.csv 3910733944a2:product.csv
docker cp sales_curated.csv 3910733944a2:sales_curated.csv
docker cp site.csv 3910733944a2:site.csv
```

We will import data to Memgraph using the `LOAD CSV` Cypher clause that enables us to load and use data from a CSV file of our choosing in a row-based manner within a query. Memgraph supports the Excel CSV dialect, the most common one. For more info, check out [Memgraph Docs](https://memgraph.com/docs/memgraph/import-data/load-csv-clause).

The following method will run the `LOAD CSV` clause inside Memgraph and create `Movie` and `Genre` nodes and the `OF_GENRE` relationship between them.

In [22]:
memgraph.execute(
    """
    LOAD CSV FROM "/site.csv" WITH HEADER AS row
    CREATE (s:Site {
    site_id       : toInteger(row.site_number), 
    store_code    : row.store_code,     
    site_name  	  : row.site_name, 
    site_address  : row.site_address, 
    lat		      : row.latitude, 
    lng		      : row.longitude
    }) 
    """
)

In [23]:
memgraph.execute(
    """
    LOAD CSV FROM "/product.csv" WITH HEADER AS row
    CREATE (a:Article {article_id: toInteger(row.article_number), 
    article_type_code         : row.article_type_code         ,
    created_date              : row.created_date              ,
    manufacturer_code         : row.manufacturer_code         ,
    gross_weight              : row.gross_weight              ,
    net_weight                : row.net_weight                ,
    speed_rating_code         : row.speed_rating_code         ,
    tire_cross_section_number : row.tire_cross_section_number ,
    tire_aspect_ratio         : row.tire_aspect_ratio         ,
    tire_rim_size_number      : row.tire_rim_size_number      ,
    tread_depth               : row.tread_depth               ,
    tire_load_capacity        : row.tire_load_capacity        ,
    tire_load_index           : row.tire_load_index           ,
    wheel_offset_number       : row.wheel_offset_number       ,
    wheel_diameter_number     : row.wheel_diameter_number     ,
    wheel_style_code          : row.wheel_style_code          ,
    wheel_bolt_pattern        : row.wheel_bolt_pattern        ,
    wheel_accent_code         : row.wheel_accent_code         ,
    spoke_100mm_depth         : row.spoke_100mm_depth         ,
    spoke_120mm_depth         : row.spoke_120mm_depth         ,
    spoke_160mm_depth         : row.spoke_160mm_depth         ,
    spoke_90mm_depth          : row.spoke_90mm_depth          ,
    merchandise_category_code : row.merchandise_category_code }) 
    """
)

In [24]:
memgraph.execute(
    """
    LOAD CSV FROM "/employee.csv" WITH HEADER AS row
    CREATE (e:Employee {employee_id: toInteger(row.employee_identifier), 
    site_number                    : row.site_number,
    full_name                      : row.full_name                    ,
    employee_type_name             : row.employee_type_name           ,
    employment_status_code         : row.employment_status_code       ,
    position_effective_start_date  : row.position_effective_start_date,
    effective_termination_date     : row.effective_termination_date   ,
    position_name                  : row.position_name                ,
    position_job_code              : row.position_job_code            ,
    original_hire_date             : row.original_hire_date           ,
    create_timestamp               : row.db_create_timestamp          
    }) 
    """
)

In [25]:
memgraph.execute(
    """
    LOAD CSV FROM "/article_inventory.csv" WITH HEADER AS row
    CREATE (i:Inventory {
    inventory_id           : toInteger(row.inventory_id),
    site_number            : toInteger(row.site_number),
    article_number         : toInteger(row.article_number),
    reserved_quantity      : toInteger(row.reserved_quantity),
    on_hand_quantity       : toInteger(row.on_hand_quantity),
    available_quantity     : toInteger(row.available_quantity),
    in_transit_quantity    : toInteger(row.in_transit_quantity),
    layaway_quantity       : toInteger(row.layaway_quantity),
    weborder_quantity      : toInteger(row.weborder_quantity),
    inventory_date         : toString(row.inventory_date)
    })
    """
)

In [26]:
memgraph.execute(
    """
    LOAD CSV FROM "/sales_curated.csv" WITH HEADER AS row
    CREATE (s:Sales {
    sales_id                      : toInteger(row.sales_order_identifier),
    employee_number               : toInteger(row.create_employee_identifier),
    article_number                : toInteger(row.article_number),
    sales_order_created_date      : toString(row.sales_order_created_date), 
    sold_quantity                 : toInteger(row.sold_quantity), 
    retail_price                  : toFloat(row.retail_price), 
    discount_amount               : toFloat(row.discount_amount), 
    net_price   				  : toFloat(row.net_price)
    })
    """
)

In [27]:
memgraph.execute(
    """
    LOAD CSV FROM "/article_inventory.csv" WITH HEADER AS row
    MATCH (s:Site {site_id: toInteger(row.site_number)})
    MATCH (i:Inventory {inventory_id: toInteger(row.inventory_id)})
    MERGE (s)-[:OwnInventory {
    r_id              : toInteger(row.site_number) + toInteger(row.inventory_id),
    site_id           : toInteger(row.site_number),
    inventory_id      : toInteger(row.inventory_id),
    create_timestamp  : toString(row.db_create_timestamp)
    }]->(i);
    """
)

In [28]:
memgraph.execute(
    """
    LOAD CSV FROM "/sales_curated.csv" WITH HEADER AS row
    MATCH (s:Sales {sales_id: toInteger(row.sales_order_identifier)})
    MATCH (e:Employee {employee_id: toInteger(row.create_employee_identifier)})
    MERGE (e)-[:CreateSales {
    r_id              : toInteger(row.sales_order_identifier) + toInteger(row.create_employee_identifier),
    sales_id          : toInteger(row.sales_order_identifier),
    employee_id       : toInteger(row.create_employee_identifier),
    create_timestamp  : toString(row.db_create_timestamp)
    }]->(s);
    """
)

In [29]:
memgraph.execute(
    """
    LOAD CSV FROM "/employee.csv" WITH HEADER AS row
    MATCH (s:Site {site_id: toInteger(row.site_number)})
    MATCH (e:Employee {employee_id: toInteger(row.employee_identifier)})
    MERGE (e)-[:BelongtoSite {
    r_id              : toInteger(row.site_number) + toInteger(row.employee_identifier),
    site_id          : toInteger(row.site_number),
    employee_id       : toInteger(row.employee_identifier),
    create_timestamp  : toString(row.db_create_timestamp)
    }]->(s);
    """
)

In [30]:
memgraph.execute(
    """
    LOAD CSV FROM "/sales_curated.csv" WITH HEADER AS row
    MATCH (s:Sales {sales_id: toInteger(row.sales_order_identifier)})
    MATCH (a:Article {article_id: toInteger(row.article_number)})
    MERGE (s)-[:SellArticle {
    r_id              : toInteger(row.sales_order_identifier) + toInteger(row.article_number),
    sales_id          : toInteger(row.sales_order_identifier),
    article_id        : toInteger(row.article_number),
    create_timestamp  : toString(row.db_create_timestamp)
    }]->(a);
    """
)

In [31]:
memgraph.execute(
    """
    LOAD CSV FROM "/article_inventory.csv" WITH HEADER AS row
    MATCH (i:Inventory {inventory_id: toInteger(row.inventory_id)})
    MATCH (a:Article {article_id: toInteger(row.article_number)})
    MERGE (i)-[:HoldArticle {
    r_id              : toInteger(row.inventory_id) + toInteger(row.article_number),
    inventory_id      : toInteger(row.sales_order_identifier),
    article_id        : toInteger(row.article_number),
    create_timestamp  : toString(row.db_create_timestamp)
    }]->(a);
    """
)

Great! We imported all the data into Memgraph. Feel free to check it out in Memgraph Lab at `localhost:3000`.

## 9. Querying the database and retrieving results <a name="query"></a>

The simplest usage of the **Cypher query language** is to find data stored in the database. For that, you can use one of the following clauses:

- `MATCH` which searches for patterns.
- `WHERE` for filtering the matched data.
- `RETURN` for defining what will be presented to the user in the result set.
- `UNION` and `UNION ALL` for combining results from multiple queries.
- `UNWIND` for unwinding a list of values as individual rows.

Let's create a simple `MATCH` query to find all movies and order them by their title in ascending order.

In [32]:
results = memgraph.execute_and_fetch(
    """
    MATCH (n:Site)
    RETURN n
    ORDER BY n.site_name ASC
    LIMIT 10;
    """
)

In [33]:
results = list(results)

for result in results:
    print(result["n"])

<Site id=479289 labels={'Site'} properties={'site_id': 2098, 'store_code': 'FLJ 10', 'site_name': '103RD ST (FLJ 10)', 'site_address': '7273 103RD ST', 'lat': 30.248608, 'lng': -81.766545}>
<Site id=478729 labels={'Site'} properties={'site_id': 1515, 'store_code': 'COD 27', 'site_name': '104TH (COD 27)', 'site_address': '3805 E 104TH AVE', 'lat': 39.885054, 'lng': -104.942006}>
<Site id=479246 labels={'Site'} properties={'site_id': 2055, 'store_code': 'KSK 03', 'site_name': '119TH ST (KSK 03)', 'site_address': '11640 METCALF AVE', 'lat': 38.915491, 'lng': -94.669252}>
<Site id=478826 labels={'Site'} properties={'site_id': 1618, 'store_code': 'TXD 60', 'site_name': '121 CUSTER (TXD 60)', 'site_address': '2260 STATE HIGHWAY 121', 'lat': 33.123562, 'lng': -96.735395}>
<Site id=478736 labels={'Site'} properties={'site_id': 1522, 'store_code': 'MID 11', 'site_name': '14 MILE RD (MID 11)', 'site_address': '1125 E 14 MILE RD#1137', 'lat': 42.534588, 'lng': -83.097343}>
<Site id=479183 labels=

We may wonder whether these results have been correctly serialized back to the defined schema. We can check that out:

In [34]:
n = results[8]["n"]

print("Site: ", n.site_name)
print("Type: ", type(n))

Site:  28TH ST (COB 01)
Type:  <class '__main__.Site'>


If you're not keen on Cypher query language, you can try out the GQLAlchemy **query builder**. Let's rewrite the above Cypher query with the help of the query builder:

In [35]:
from gqlalchemy import match
from gqlalchemy.query_builders.memgraph_query_builder import Order

results_from_qb = (
    match()
    .node(labels="Site", variable="s")
    .return_()
    .order_by(("s.site_name", Order.ASC))
    .limit(10)
    .execute()
)
results_from_qb = list(results_from_qb)

for result in results_from_qb:
    print(result["s"])

<Site id=479289 labels={'Site'} properties={'site_id': 2098, 'store_code': 'FLJ 10', 'site_name': '103RD ST (FLJ 10)', 'site_address': '7273 103RD ST', 'lat': 30.248608, 'lng': -81.766545}>
<Site id=478729 labels={'Site'} properties={'site_id': 1515, 'store_code': 'COD 27', 'site_name': '104TH (COD 27)', 'site_address': '3805 E 104TH AVE', 'lat': 39.885054, 'lng': -104.942006}>
<Site id=479246 labels={'Site'} properties={'site_id': 2055, 'store_code': 'KSK 03', 'site_name': '119TH ST (KSK 03)', 'site_address': '11640 METCALF AVE', 'lat': 38.915491, 'lng': -94.669252}>
<Site id=478826 labels={'Site'} properties={'site_id': 1618, 'store_code': 'TXD 60', 'site_name': '121 CUSTER (TXD 60)', 'site_address': '2260 STATE HIGHWAY 121', 'lat': 33.123562, 'lng': -96.735395}>
<Site id=478736 labels={'Site'} properties={'site_id': 1522, 'store_code': 'MID 11', 'site_name': '14 MILE RD (MID 11)', 'site_address': '1125 E 14 MILE RD#1137', 'lat': 42.534588, 'lng': -83.097343}>
<Site id=479183 labels=

As expected, the result is the same as above.

## 10. Use Case queries  <a name="exercises"></a>


### 🔷 Use Case 1: Find out how many Articles there are in the database.

The correct Cypher query would be:

```
MATCH (n:Article)
RETURN count(n) AS num_of_article;
```

You can try it out in Memgraph Lab at `localhost:3000`.

With GQLAlchemy's query builder, the solution is:

In [36]:
total_users = (
    match()
    .node(labels="Article", variable="u")
    .return_(("count(u)", "num_of_article"))
    .execute()
)

results = list(total_users)
for result in results:
    print(result["num_of_article"])

50000


### 🔷 Use Case 2: Find out how many Sales sold the Article with the type code 'ZTIR'.

The correct Cypher query would be:

```
MATCH (s:Sales)-[:SoldArticle]->(:Article {article_type_code: 'ZTIR'})
RETURN count(s) AS num_of_sales;
```

You can try it out in Memgraph Lab at `localhost:3000`.

With GQLAlchemy's query builder, the solution is:

In [37]:
from gqlalchemy.query_builders.memgraph_query_builder import Operator

sold_alone = (
    match()
    .node(labels="Sales", variable="s")
    .to(relationship_type="SellArticle")
    .node(labels="Article", variable="a")
    .where(item="a.article_type_code", operator=Operator.EQUAL, literal="ZTIR")
    .return_(("count(s)", "num_of_sales"))
    .execute()
)


results = list(sold_alone)

for result in results:
    print(result["num_of_sales"])

8286


### 🔷 Use Case 3: Find out the biggest sales for a type of Article.

The correct Cypher query would be:

```
MATCH (s:Sales)-[r:SellArticle]->(a:Article{article_type_code: 'ZTIR'})
RETURN s.sold_quantity * s.retail_price as total order by total DESCENDING limit 1;
```

You can try it out in Memgraph Lab at `localhost:3000`.

With GQLAlchemy's query builder, the solution is:

In [38]:
from gqlalchemy.query_builders.memgraph_query_builder import Operator

employee_alone = (
    match()
    .node(labels="Sales", variable="s")
    .to(relationship_type="SellArticle", variable="r")
    .node(labels="Article", variable="a")
    .where(item="a.article_type_code", operator=Operator.EQUAL, literal="ZTIR")
    .return_(("max(s.sold_quantity * s.retail_price)", "total"))
    .execute()
)

results = list(employee_alone)

for result in results:
    print(result["total"])

15288.0


### 🔷 Clean up.

```
MATCH (n)
DETACH DELETE n
```

```
MATCH ()
RETURN count (*)
```



<h4 align="center"><a href=#toc>⬆️ GO TO TOP ⬆️</a></h3> 

In [39]:
import sqlalchemy
sqlalchemy.__version__

'1.4.52'